# Compute a transfer function

Atmospheric emission can dominate the noise budget and must be removed from the time-ordered data before mapping. This filtering inevitably projects out correlated sky modes, suppressing signal at large angular scales. The spatial transfer function $T(u)$ provides a direct estimate of this scale-dependent signal loss. `maria` includes a built-in function to compute this, using the cross-spectrum between the output and the known input map:

$$T(u) = \frac{\mathrm{Re}\langle \tilde{s}_{\rm in}^*(u)\,\tilde{s}_{\rm out}(u)\rangle}{\langle|\tilde{s}_{\rm in}(u)|^2\rangle}$$

Please note that the cross-spectrum is used instead of the power ratio $P_{\rm out}/P_{\rm in}$ because noise in the output is uncorrelated with the input and averages to zero, leaving an unbiased estimate.

## Input sky

In this tutorial, we simulate a two-frequency observation of a galaxy cluster at 150 and 270 GHz and recover a map with the `BinMapper`. We load the same map at both frequencies — any difference in the recovered signal will come from the instrument and the filtering alone.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import maria
from maria.io import fetch

map_filename = fetch("maps/cluster2.fits")

nu = np.array([150e9, 270e9])

m1 = maria.map.load(filename=map_filename, nu=nu[0], width=20.00/60.00)
m2 = maria.map.load(filename=map_filename, nu=nu[1], width=20.00/60.00)

# Arbitrary signal boost to improve the SNR of the output map
m1.data *= -5.0e3
m2.data *= -5.0e3

maps = maria.map.concatenate([m1, m2], dim="nu")
print(maps)
maps.to("K_RJ").plot(slices=dict(nu=[0, 1]), cmap="RdYlBu_r")

## Instrument and observation plan

We use TolTEC on the LMT, keeping only the 150 and 270 GHz arrays and dropping the 220 GHz one for efficiency. Because the beam is diffraction-limited, the 270 GHz array resolves angular scales roughly 1.8× finer than the 150 GHz array.

In [ ]:
from maria.instrument import get_instrument_config, Instrument

config = get_instrument_config("TolTEC")
config["arrays"] = {k: config["arrays"][k] for k in ["array-1", "array-3"]}
instrument = Instrument.from_config(config)

print(instrument)
instrument.plot()

In [ ]:
from maria import Planner

planner = Planner(
    target=maps,
    start_time="2024-01-01T22:00:00",
    site="llano_de_chajnantor",
    constraints={"el": (60, 90)},
)

plans = planner.generate_plans(
    total_duration=0.1 * 3600,
    max_chunk_duration=3600,
    scan_options={"radius": 6.5 / 60.0, "miss_factor": 0.3},
)

print(plans)
plans[0].plot()

## Simulation

Passing `map=maps` attaches the ground-truth sky to each output TOD so the mapper can propagate it to the output map automatically.

In [ ]:
sim = maria.Simulation(
    instrument,
    plans=plans,
    site="llano_de_chajnantor",
    map=maps,
    atmosphere="2d",
)

print(sim)

tods = sim.run()
tods[0].plot()

## Mapmaking

Common-mode subtraction (`remove_modes`) and per-detector spline removal (`remove_spline`) suppress large-scale correlated signal. Both operations remove power at long time scales, which maps to large angular scales on the sky, i.e., where $T$ will fall below 1.

> **Note:** `map_postprocessing` is intentionally left empty here. Any map-level post-processing (smoothing, filtering, etc.) applied after binning would itself suppress or modify signal in the output map, and its effect would be absorbed into the measured $T(u)$. To isolate the transfer function of the TOD filtering alone, always set `map_postprocessing={}` when computing transfer functions.

In [ ]:
from maria.mappers import BinMapper

mapper = BinMapper(
    tods=tods,
    units="uK_RJ",
    stokes="I",
    resolution=maps.resolution,
    tod_preprocessing={
        "remove_modes": {"modes_to_remove": 1},
        "remove_spline": {"knot_spacing": 60, "remove_el_gradient": True},
    },
    map_postprocessing={},
)

output_map = mapper.run()

In [ ]:
output_map.plot(slices=dict(nu=[0, 1]), cmap="RdYlBu_r")

## Transfer function

To compute the transfer function, call `transfer_function()` on the output map. This will return a `TransferFunction` object containing the transfer function $T(u)$ and the corresponding spatial frequencies $u$. The transfer function can be plotted to visualize the scale-dependent signal loss.

In [ ]:
tf = output_map.transfer_function(window=True)
print(tf)

To visualize the transfer function, it is possible to use the built-in `plot()` method. Solid curves show the measured $T(u)$, while dashed curves show the Gaussian beam per channel. The large-scale rolloff reflects the TOD filtering above, while the small-scale rolloff tracks the beam.

In [ ]:
tf.plot(x_unit="arcmin")

Finally, individual channels can be selected with `slices=dict(nu=[...])`, consistent with how `slices` is used in `.plot()`:

In [ ]:
tf.plot(slices=dict(nu=[0]), x_unit="arcmin")